# H-008 Baseline Experiment

**WP-4b2** | H-008: Premium-Confirmed Momentum Continuation | 2026-04-17

Primary timeframe: 1h | Free parameters: 4 | Trial ID: `20605a84-41ec-4d41-860d-8450698fbf69`

Evaluation note: no separate IS/OOS split convention exists in repo truth. The baseline uses the full registered evaluation range for both `in_sample_sharpe` and `oos_sharpe`, matching the current runbook baseline pattern.

Gate 2 semantics note: this notebook reports Gate 2 according to current `lib/evaluation.py` repo truth. The displayed legacy `GateReport` wording may read `DSR > 1.0 at 95% CI`, but the evaluator currently checks whether `dsr_val` exceeds 0.95 against its expected maximum-SR benchmark from the registered trial count.

Premium gap note: the 1h premium index stream has one known irreparable bar (2020-12-01) per `data/irreparable_gaps.json`. This produces a single NaN in premium alignment and propagates to a NaN in `premium_z` for that bar. The signal loop handles NaN bars by setting signal to 0 (flat), satisfying the D2-003A explicit-handling requirement.

In [ ]:
from pathlib import Path
import json
import hashlib
import math
import pandas as pd
import numpy as np
from lib.backtest import run_backtest
from lib.evaluation import check_promotion_gates, cost_drag, hit_rate, max_drawdown, profit_factor, sharpe

HYPOTHESIS_ID = "H-008"
TRIAL_ID = "20605a84-41ec-4d41-860d-8450698fbf69"
TRIAL_COUNT_BEFORE_APPEND = 8
DATASET_IDS = [
    "binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_premium_index_1h_2019-12-24_2026-04-11_2026-04-13T0136Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z",
]
PARAMS = {"momentum_lookback": 24, "premium_lookback": 168, "premium_z_threshold": 1.0, "exit_lookback": 6}
COSTS = {"fee_bps_taker": 4.0, "fee_bps_maker": 2.0, "slippage_bps": 2.0, "initial_equity": 10000.0}

DATA_ROOT = Path("data/processed")
OHLCV_1H_PATH   = DATA_ROOT / "binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet"
PREMIUM_1H_PATH = DATA_ROOT / "binance_um_perp_BTCUSDT_premium_index_1h_2019-12-24_2026-04-11_2026-04-13T0136Z.parquet"
FUNDING_PATH    = DATA_ROOT / "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet"

def _sha256_of_dict(d):
    return hashlib.sha256(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    return df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()

def load_kline(path, name):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    out = df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()
    out.columns = [f"{name}_{col}" for col in out.columns]
    return out

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df["fundingTime"].astype("int64"), unit="ms", utc=True)
    return df.rename(columns={"fundingRate": "funding_rate"})[["funding_rate"]].astype(float).sort_index()

def filter_applicable_funding(signal, ohlcv, funding):
    common_idx = signal.index.intersection(ohlcv.index)
    sig = signal.reindex(common_idx).fillna(0).astype(int)
    bars = list(common_idx)
    position = 0
    event_ts = []
    funding_idx = funding.index
    for i, bar_ts in enumerate(bars):
        if i > 0 and position != 0:
            prev_bar_ts = bars[i - 1]
            matches = funding_idx[(funding_idx > prev_bar_ts) & (funding_idx <= bar_ts)]
            event_ts.extend(matches)
        if i > 0:
            desired = int(sig.iloc[i - 1])
            if desired != position:
                position = desired
    if not event_ts:
        return funding.iloc[0:0].copy()
    return funding.loc[pd.DatetimeIndex(sorted(set(event_ts)))]

def run_candidate(signal, ohlcv, funding):
    applicable = filter_applicable_funding(signal, ohlcv, funding)
    result = run_backtest(signal, ohlcv, applicable, **COSTS, execution="next_bar_open", stream="klines", interval="1h")
    return result, len(applicable)

def summarize(result):
    returns = result.equity_curve.pct_change().dropna()
    mdd, peak, trough = max_drawdown(result.equity_curve)
    cd = cost_drag(result)
    return {"final_equity": float(result.equity_curve.iloc[-1]), "total_pnl": float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]), "sharpe": float(sharpe(returns, "1h")), "max_drawdown": float(mdd), "max_drawdown_peak": str(peak) if peak is not None else None, "max_drawdown_trough": str(trough) if trough is not None else None, "n_obs": int(len(returns)), "n_trades": int(len(result.trades_df)), "hit_rate": float(hit_rate(result.trades_df)), "profit_factor": float(profit_factor(result.trades_df)), "fees_paid": float(result.fees_paid), "slippage_paid": float(result.slippage_paid), "funding_paid": float(result.funding_paid), "gross_pnl": float(cd["gross_pnl"]), "cost_drag_fraction": float(cd["cost_drag_fraction"]) if math.isfinite(cd["cost_drag_fraction"]) else None, "skipped_bars": int(result.skipped_bars)}

def side_summary(result):
    if result.trades_df.empty:
        return []
    return [{"direction": int(d), "count": int(len(g)), "net_pnl_sum": float(g["net_pnl"].sum()), "net_pnl_mean": float(g["net_pnl"].mean())} for d, g in result.trades_df.groupby("direction")]

def sparkline(series, width=72):
    vals = series.dropna()
    if vals.empty: return ""
    if len(vals) > width: vals = vals.iloc[::max(1, len(vals) // width)].iloc[:width]
    chars = "._-~=+*#%@"
    lo, hi = float(vals.min()), float(vals.max())
    if hi == lo: return chars[0] * len(vals)
    return "".join(chars[min(len(chars) - 1, int((float(v) - lo) / (hi - lo) * (len(chars) - 1)))] for v in vals)

def drawdown_series(equity):
    return (equity - equity.cummax()) / equity.cummax()

def h008_signal(ohlcv, premium_close, momentum_lookback=24, premium_lookback=168,
                premium_z_threshold=1.0, exit_lookback=6):
    close = ohlcv["close"]
    ml = int(round(momentum_lookback)); pl = int(round(premium_lookback)); el = int(round(exit_lookback)); z_thresh = float(premium_z_threshold)
    price_momentum = close.pct_change(ml)
    roll_mean = premium_close.rolling(pl, min_periods=pl).mean()
    roll_std  = premium_close.rolling(pl, min_periods=pl).std()
    premium_z = (premium_close - roll_mean) / roll_std
    exit_momentum = close.pct_change(el)
    signal = pd.Series(0.0, index=ohlcv.index, name="signal")
    current = 0.0
    for ts in ohlcv.index:
        pm = price_momentum.loc[ts]; pz = premium_z.loc[ts]; em = exit_momentum.loc[ts]
        if pd.isna(pm) or pd.isna(pz) or pd.isna(em):
            current = 0.0; signal.loc[ts] = 0.0; continue
        long_entry = (pm > 0) and (pz > z_thresh); short_entry = (pm < 0) and (pz < -z_thresh)
        exit_long = (pz <= 0) or (em < 0); exit_short = (pz >= 0) or (em > 0)
        if current == 1.0:
            if exit_long: current = -1.0 if short_entry else 0.0
        elif current == -1.0:
            if exit_short: current = 1.0 if long_entry else 0.0
        else:
            if long_entry: current = 1.0
            elif short_entry: current = -1.0
        signal.loc[ts] = current
    return signal

def sensitivity_h008(ohlcv, premium_close, funding, base_sharpe):
    base = PARAMS.copy()
    variants = [("momentum_lookback_up25", {**base, "momentum_lookback": int(round(24*1.25))}), ("momentum_lookback_dn25", {**base, "momentum_lookback": int(round(24*0.75))}), ("premium_lookback_up25", {**base, "premium_lookback": int(round(168*1.25))}), ("premium_lookback_dn25", {**base, "premium_lookback": int(round(168*0.75))}), ("premium_z_threshold_up25", {**base, "premium_z_threshold": 1.25}), ("premium_z_threshold_dn25", {**base, "premium_z_threshold": 0.75}), ("exit_lookback_up25", {**base, "exit_lookback": int(round(6*1.25))}), ("exit_lookback_dn25", {**base, "exit_lookback": int(round(6*0.75))})]
    rows = []
    for name, params in variants:
        sig = h008_signal(ohlcv, premium_close, **params)
        res, _ = run_candidate(sig, ohlcv, funding)
        sr = summarize(res)["sharpe"]
        rows.append({"variant": name, "params": params, "sharpe": sr, "ratio_to_base": (sr / base_sharpe) if base_sharpe and base_sharpe != 0 else 0.0})
    return rows

ohlcv = load_ohlcv(OHLCV_1H_PATH)
premium = load_kline(PREMIUM_1H_PATH, "premium")
funding = load_funding()

common_start = max(ohlcv.index.min(), premium.index.min())
ohlcv = ohlcv.loc[common_start:]
premium_close = premium["premium_close"].reindex(ohlcv.index)

signal = h008_signal(ohlcv, premium_close, **PARAMS)
roll_mean = premium_close.rolling(168, min_periods=168).mean()
roll_std  = premium_close.rolling(168, min_periods=168).std()
premium_z_series = (premium_close - roll_mean) / roll_std
premium_z_valid = int(premium_z_series.notna().sum())
premium_confirmation_pct = float((premium_z_series.abs() > 1.0).sum() / premium_z_valid * 100) if premium_z_valid > 0 else 0.0

result, funding_events = run_candidate(signal, ohlcv, funding)
metrics = summarize(result)
sensitivity = sensitivity_h008(ohlcv, premium_close, funding, metrics["sharpe"])
sensitivity_min_ratio = min(r["ratio_to_base"] for r in sensitivity)

params_meta = {"n_free_params": 4, "sensitivity_min_ratio": sensitivity_min_ratio, "is_sharpe": metrics["sharpe"]}
gate_report = check_promotion_gates(result=result, trial_count=TRIAL_COUNT_BEFORE_APPEND, params_meta=params_meta, freq="1h")

config = {"hypothesis_id": HYPOTHESIS_ID, "primary_timeframe": "1h", "execution": "next_bar_open", "costs": COSTS, "params": PARAMS}
report = {"hypothesis_id": HYPOTHESIS_ID, "trial_id": TRIAL_ID, "trial_count_before_append": TRIAL_COUNT_BEFORE_APPEND, "dataset_ids": DATASET_IDS, "dataset_sha256": _sha256_of_dict({"ids": sorted(DATASET_IDS)}), "config_sha256": _sha256_of_dict(config), "params": PARAMS, "funding_events_applied": funding_events, "signal_counts": {str(k): int(v) for k, v in signal.value_counts().sort_index().items()}, "premium_z_valid_bars": premium_z_valid, "premium_confirmation_pct": premium_confirmation_pct, "metrics": metrics, "sensitivity": sensitivity, "sensitivity_min_ratio": float(sensitivity_min_ratio), "side_summary": side_summary(result), "equity_sparkline": sparkline(result.equity_curve), "drawdown_sparkline": sparkline(drawdown_series(result.equity_curve)), "gate_report": gate_report.to_dict(), "verdict": "PASS" if gate_report.overall_pass else "NO-GO"}
print(json.dumps(report, indent=2, sort_keys=True))

{
  "config_sha256": "800df404f492dc4dfee866ace66d378b95eb18f47ed902c81aedb4b2911a7a84",
  "dataset_sha256": "87a2754362bbd5f91ed8d19c00d2bdcca0c6f3ffba0187e3dc72651f36d2e468",
  "drawdown_sparkline": "@%%%%%%%%%%%%%%%%%####*********++++++++++++++++======~~~~~------_____...",
  "equity_sparkline": "@%%%%%%%%%%%%%%%%%####*********++++++++++++++++======~~~~~------_____...",
  "funding_events_applied": 1407,
  "gate_report": {
    "gates": [
      {"gate_id": 1, "gate_name": "OOS Sharpe >= 1.0", "passed": false, "threshold": 1.0, "value": -0.1502},
      {"gate_id": 2, "gate_name": "DSR > 1.0 at 95% CI", "note": "P(SR>1.0)=0.0000; SR benchmark=0.4614", "passed": false, "threshold": 1.0, "value": 0.0},
      {"gate_id": 3, "gate_name": "Per-fold max DD <= 20%", "passed": false, "threshold": 0.2, "value": 39.2885},
      {"gate_id": 4, "gate_name": "Parameter sensitivity min_ratio >= 0.70", "passed": false, "threshold": 0.7, "value": -2.9706},
      {"gate_id": 5, "gate_name": "Cost drag <=

## Verdict

**NO-GO** — 3 of 8 gates passed (Gates 5, 6, 7).

### Key Observations

| Metric | H-008 | H-005 (reference) |
|--------|-------|-------------------|
| Sharpe (full-period) | -0.1502 | 0.3751 |
| Max Drawdown | 3928.85% | 212.08% |
| Total Trades | 5,211 | 934 |
| Fees Paid | $203,860 | $38,150 |
| Cost Drag | -338.05% (mech. pass) | 445.53% |
| Gross PnL | -$91,802 | $13,184 |

Premium confirmation frequency: **30.4%** of valid bars had |premium_z| > 1.0. Despite this selectivity, trade count (5211) was **5.6x higher than H-005** (934 trades). The exit momentum condition (exit_lookback=6) caused rapid re-entries after exits, churning through positions. Gross PnL was negative (-$91,802) before fees, meaning the signal had no edge to begin with. Premium z-score confirmation did not improve on the price-only momentum quality of H-005.

Gates failed: Sharpe (G1), DSR (G2), Max DD (G3), Sensitivity (G4), Walk-Forward Efficiency (G8).

Decision record: [DEC-0008](../../../decisions/DEC-0008.md)